In [13]:
"""
Data Preparation for Optimal Execution
======================================

"""

import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from pathlib import Path
import gc
import joblib

# ====================== CONFIG ======================
data_dir = Path('/content/drive/MyDrive/spydaytrading')

parquet_files = sorted(data_dir.glob('spy_*_market_hours.parquet'))
print(f"Found {len(parquet_files)} files")

bin_rule = '10s'
n_train_files = 18
train_files = parquet_files[:n_train_files]
test_files = parquet_files[n_train_files:]

# Column definitions (same as before)
bid_px_cols = [f'bid_px_{i:02d}' for i in range(10)]
ask_px_cols = [f'ask_px_{i:02d}' for i in range(10)]
bid_sz_cols = [f'bid_sz_{i:02d}' for i in range(10)]
ask_sz_cols = [f'ask_sz_{i:02d}' for i in range(10)]
price_cols = bid_px_cols + ask_px_cols
size_cols = bid_sz_cols + ask_sz_cols

# Accumulators - MODIFIED to include price/volume
all_data = []

# ====================== PROCESS ALL FILES ======================
all_files = train_files + test_files
is_training_file = [True] * len(train_files) + [False] * len(test_files)

for idx, (file_path, is_train) in enumerate(zip(all_files, is_training_file)):
    status = "TRAIN" if is_train else "TEST "
    print(f"[{status}] [{idx+1}/{len(all_files)}] Processing {file_path.name}")

    # Load ALL needed columns including prices
    df = pd.read_parquet(file_path, columns=price_cols + size_cols)

    if 'ts_event' in df.columns:
        df = df.set_index('ts_event')
    elif 'ts_recv' in df.columns:
        df = df.set_index('ts_recv')
    df = df.sort_index()

    # =========================================================
    # NEW: Calculate market microstructure features per event
    # =========================================================

    # Mid price (best bid + best ask) / 2
    df['mid_price'] = (df['bid_px_00'] + df['ask_px_00']) / 2

    # Spread
    df['spread'] = df['ask_px_00'] - df['bid_px_00']

    # Microprice (size-weighted mid)
    df['microprice'] = (
        df['bid_px_00'] * df['ask_sz_00'] + df['ask_px_00'] * df['bid_sz_00']
    ) / (df['bid_sz_00'] + df['ask_sz_00'])

    # Total visible depth (all levels)
    df['total_bid_size'] = df[bid_sz_cols].sum(axis=1)
    df['total_ask_size'] = df[ask_sz_cols].sum(axis=1)
    df['total_depth'] = df['total_bid_size'] + df['total_ask_size']

    # Book imbalance (top of book)
    df['book_imbalance'] = (df['bid_sz_00'] - df['ask_sz_00']) / (df['bid_sz_00'] + df['ask_sz_00'])

    # =========================================================
    # OFI Calculation (same as your original)
    # =========================================================
    df_lag = df[price_cols + size_cols].shift(1)
    df_lag.columns = [c + '_lag' for c in df_lag.columns]
    data = pd.concat([df, df_lag], axis=1).dropna()

    ofi_levels = []
    for i in range(10):
        b_px = f'bid_px_{i:02d}'
        a_px = f'ask_px_{i:02d}'
        b_sz = f'bid_sz_{i:02d}'
        a_sz = f'ask_sz_{i:02d}'
        b_px_lag = b_px + '_lag'
        a_px_lag = a_px + '_lag'
        b_sz_lag = b_sz + '_lag'
        a_sz_lag = a_sz + '_lag'

        bid_of = np.where(
            data[b_px] == data[b_px_lag],
            data[b_sz] - data[b_sz_lag],
            0
        )
        ask_of = np.where(
            data[a_px] == data[a_px_lag],
            -(data[a_sz] - data[a_sz_lag]),
            0
        )
        ofi_levels.append(bid_of + ask_of)

    ofi_event = pd.DataFrame(
        np.column_stack(ofi_levels),
        columns=[f'OFI_{i}' for i in range(10)],
        index=data.index
    )

    # =========================================================
    # Bin Aggregation - MODIFIED to include more features
    # =========================================================

    # OFI: sum per bin
    ofi_sum_bin = ofi_event.resample(bin_rule).sum()

    # Prices: OHLC per bin
    price_ohlc = data['mid_price'].resample(bin_rule).agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last',
        'mid_price': 'last'  # Closing mid
    })

    # Spread: average per bin
    spread_bin = data['spread'].resample(bin_rule).mean()

    # Microprice: close per bin
    microprice_bin = data['microprice'].resample(bin_rule).last()

    # Depth: sum per bin (proxy for volume)
    depth_bin = data['total_depth'].resample(bin_rule).sum()

    # Book imbalance: mean per bin
    imbalance_bin = data['book_imbalance'].resample(bin_rule).mean()

    # Event count
    event_count_bin = data['mid_price'].resample(bin_rule).count()

    # Total size for normalization
    total_size_event = data[size_cols].sum(axis=1)
    size_sum_bin = total_size_event.resample(bin_rule).sum()

    # Combine into single DataFrame
    bin_data = pd.DataFrame({
        'mid_price': price_ohlc['mid_price'],
        'open': price_ohlc['open'],
        'high': price_ohlc['high'],
        'low': price_ohlc['low'],
        'close': price_ohlc['close'],
        'spread': spread_bin,
        'microprice': microprice_bin,
        'volume': depth_bin,  # Using depth as volume proxy
        'book_imbalance': imbalance_bin,
        'event_count': event_count_bin,
        'size_sum': size_sum_bin,
    })

    # Add OFI columns
    for col in ofi_sum_bin.columns:
        bin_data[col] = ofi_sum_bin[col]

    # Mark train/test
    bin_data['is_train'] = is_train
    bin_data['date'] = file_path.stem.split('_')[1]  # Extract date from filename

    all_data.append(bin_data)

    del df, data, ofi_event
    gc.collect()

# ====================== COMBINE ALL DATA ======================
print("\n=== Combining all data ===")
full_data = pd.concat(all_data).sort_index()
print(f"Total bins: {len(full_data):,}")

# ====================== NORMALIZE OFI AND COMPUTE INTEGRATED ======================
print("\n=== Computing normalized OFI ===")

# Split train/test
train_data = full_data[full_data['is_train']].copy()
test_data = full_data[~full_data['is_train']].copy()

# Normalization factor Q (average depth per event)
ofi_cols = [f'OFI_{i}' for i in range(10)]

train_Q = train_data['size_sum'] / train_data['event_count'].replace(0, np.nan)
test_Q = test_data['size_sum'] / test_data['event_count'].replace(0, np.nan)
full_Q = full_data['size_sum'] / full_data['event_count'].replace(0, np.nan)

# Normalize OFI
train_ofi_norm = train_data[ofi_cols].div(train_Q, axis=0)
train_ofi_norm = train_ofi_norm.replace([np.inf, -np.inf], np.nan).dropna(how='all')

# Fit PCA on training only
print("\n=== Fitting PCA on training data ===")
pca = PCA(n_components=1)
pca.fit(train_ofi_norm.fillna(0).values)

weights = pca.components_[0]
weights = weights / np.sum(np.abs(weights))

print("\nPCA Weights:")
for i, w in enumerate(weights):
    print(f"  Level {i}: {w:+.6f}")
print(f"Variance explained: {pca.explained_variance_ratio_[0]:.1%}")

# Apply to all data
full_ofi_norm = full_data[ofi_cols].div(full_Q, axis=0)
full_ofi_norm = full_ofi_norm.replace([np.inf, -np.inf], np.nan).fillna(0)
full_data['ofi_integrated'] = full_ofi_norm.values @ weights

# ====================== SAVE ======================
print("\n=== Saving data ===")

# Save full dataset with all features
full_data.to_parquet(data_dir / 'SPY_execution_data_full.parquet')

# Save train/test splits
train_out = full_data[full_data['is_train']].copy()
test_out = full_data[~full_data['is_train']].copy()

train_out.to_parquet(data_dir / 'SPY_execution_data_train.parquet')
test_out.to_parquet(data_dir / 'SPY_execution_data_test.parquet')

# Save PCA weights
joblib.dump(weights, data_dir / 'pca_ofi_weights.pkl')

print(f"\nSaved:")
print(f"  Full data: {len(full_data):,} bins")
print(f"  Train: {len(train_out):,} bins ({train_out['date'].nunique()} days)")
print(f"  Test: {len(test_out):,} bins ({test_out['date'].nunique()} days)")

# Print summary
print("\n=== Data Summary ===")
print(full_data[['mid_price', 'spread', 'volume', 'ofi_integrated', 'book_imbalance']].describe())

print("\n=== Columns Available ===")
print(full_data.columns.tolist())

print("\nDone! Data ready for optimal execution backtesting.")

Found 22 files
[TRAIN] [1/22] Processing spy_20251001_market_hours.parquet
[TRAIN] [2/22] Processing spy_20251002_market_hours.parquet
[TRAIN] [3/22] Processing spy_20251003_market_hours.parquet
[TRAIN] [4/22] Processing spy_20251006_market_hours.parquet
[TRAIN] [5/22] Processing spy_20251007_market_hours.parquet
[TRAIN] [6/22] Processing spy_20251008_market_hours.parquet
[TRAIN] [7/22] Processing spy_20251009_market_hours.parquet
[TRAIN] [8/22] Processing spy_20251010_market_hours.parquet
[TRAIN] [9/22] Processing spy_20251013_market_hours.parquet
[TRAIN] [10/22] Processing spy_20251014_market_hours.parquet
[TRAIN] [11/22] Processing spy_20251015_market_hours.parquet
[TRAIN] [12/22] Processing spy_20251016_market_hours.parquet
[TRAIN] [13/22] Processing spy_20251017_market_hours.parquet
[TRAIN] [14/22] Processing spy_20251020_market_hours.parquet
[TRAIN] [15/22] Processing spy_20251021_market_hours.parquet
[TRAIN] [16/22] Processing spy_20251022_market_hours.parquet
[TRAIN] [17/22] Pr

In [18]:
"""
Volume-Aware OFI Execution Strategies
=====================================

"""

import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

data_dir = Path('/content/drive/MyDrive/spydaytrading')
S = 100_000

print(f"Target: Buy {S:,} shares")


# ====================== LOAD DATA ======================
def load_data():
    df = pd.read_parquet(data_dir / 'SPY_execution_data_full.parquet')
    df['ofi'] = df['ofi_integrated']
    if df['date'].dtype == 'object':
        df['date'] = pd.to_datetime(df['date']).dt.date
    return df


# ====================== MODELS ======================
class ImpactModel:
    def __init__(self, gamma=0.5):
        self.gamma = gamma
        self.eta = 0.1
        self.sigma = 0.0001

    def fit(self, df):
        returns = df['mid_price'].pct_change().dropna()
        self.sigma = returns.std()
        avg_spread_pct = (df['spread'] / df['mid_price']).mean()
        self.eta = (0.5 * avg_spread_pct) / (self.sigma * (0.01 ** self.gamma))
        return self

    def cost_per_share(self, x, volume, ofi_mult=1.0):
        if volume <= 0 or x <= 0:
            return 0
        participation = x / volume
        return self.eta * self.sigma * (participation ** self.gamma) * ofi_mult


class OFISignal:
    def __init__(self, lambda_ofi=0.5):
        self.lambda_ofi = lambda_ofi
        self.mean = 0
        self.std = 1

    def fit(self, ofi):
        self.mean = ofi.mean()
        self.std = ofi.std()
        return self

    def cost_multiplier(self, ofi):
        z = (ofi - self.mean) / self.std
        return np.clip(np.exp(self.lambda_ofi * z), 0.5, 2.0)

    def allocation_weight(self, ofi):
        z = (ofi - self.mean) / self.std
        return np.exp(-self.lambda_ofi * z)


# ====================== STRATEGIES ======================

def twap(df, S):
    return np.ones(len(df)) * S / len(df)


def vwap(df, S):
    v = df['volume'].values
    return S * v / v.sum()


def ofi_vwap_original(df, S, ofi_signal, alpha=0.3):
    """Original (broken) OFI-VWAP: additive blend"""
    v = df['volume'].values
    ofi = df['ofi'].values

    vwap_w = v / v.sum()
    ofi_w = ofi_signal.allocation_weight(ofi)
    ofi_w = ofi_w / ofi_w.sum()

    # Additive blend - THIS IS THE PROBLEM
    w = (1 - alpha) * vwap_w + alpha * ofi_w
    return S * w / w.sum()


def ofi_vwap_multiplicative(df, S, ofi_signal, alpha=0.5):
    """
    Fixed: Multiplicative OFI adjustment on top of VWAP

    Weight = VWAP_weight * OFI_adjustment

    This preserves volume weighting while tilting within volume
    """
    v = df['volume'].values
    ofi = df['ofi'].values

    vwap_w = v / v.sum()

    # OFI adjustment factor (centered at 1.0)
    ofi_adj = ofi_signal.allocation_weight(ofi)
    ofi_adj = ofi_adj / ofi_adj.mean()  # Center at 1.0

    # Dampen the adjustment
    ofi_adj = 1 + alpha * (ofi_adj - 1)

    # Multiplicative: volume first, then OFI tilt
    w = vwap_w * ofi_adj
    return S * w / w.sum()


def ofi_vwap_volume_floor(df, S, ofi_signal, alpha=0.5, volume_percentile=50):
    """
    Fixed: Only apply OFI signal when volume > median

    In low volume periods, just use VWAP weight
    """
    v = df['volume'].values
    ofi = df['ofi'].values

    vwap_w = v / v.sum()

    # Volume threshold
    vol_threshold = np.percentile(v, volume_percentile)
    high_vol_mask = v >= vol_threshold

    # OFI adjustment only in high volume periods
    ofi_adj = np.ones(len(df))
    ofi_adj[high_vol_mask] = ofi_signal.allocation_weight(ofi[high_vol_mask])
    ofi_adj[high_vol_mask] = ofi_adj[high_vol_mask] / ofi_adj[high_vol_mask].mean()

    # Dampen
    ofi_adj = 1 + alpha * (ofi_adj - 1)

    w = vwap_w * ofi_adj
    return S * w / w.sum()


def ofi_vwap_volume_scaled(df, S, ofi_signal, alpha=0.5):
    """
    Fixed: Scale OFI influence by volume

    High volume + good OFI = strong signal
    Low volume + good OFI = weak signal (can't trust it)
    """
    v = df['volume'].values
    ofi = df['ofi'].values

    vwap_w = v / v.sum()

    # Volume scaling factor (0 to 1)
    vol_scale = (v - v.min()) / (v.max() - v.min())

    # OFI adjustment
    ofi_raw = ofi_signal.allocation_weight(ofi)
    ofi_raw = ofi_raw / ofi_raw.mean() - 1  # Center at 0

    # Scale OFI by volume: stronger signal in high volume
    ofi_adj = 1 + alpha * ofi_raw * vol_scale

    w = vwap_w * ofi_adj
    return S * w / w.sum()


def ofi_opportunistic(df, S, ofi_signal, ofi_threshold=0.5, max_participation=0.05):
    """
    Opportunistic: Only buy when OFI is favorable AND volume is good

    Otherwise, spread remaining shares with VWAP
    """
    v = df['volume'].values
    ofi = df['ofi'].values
    n = len(df)

    # Normalize OFI
    ofi_z = (ofi - ofi_signal.mean) / ofi_signal.std

    # Find "good" periods: negative OFI (favorable) AND high volume
    vol_percentile = np.percentile(v, 60)
    good_periods = (ofi_z < -ofi_threshold) & (v > vol_percentile)

    allocation = np.zeros(n)

    # In good periods: allocate up to max_participation of volume
    if good_periods.sum() > 0:
        good_capacity = np.minimum(v[good_periods] * max_participation, S / good_periods.sum())
        total_good = good_capacity.sum()

        if total_good >= S:
            # Enough capacity in good periods
            allocation[good_periods] = good_capacity * S / total_good
        else:
            # Fill good periods, then VWAP the rest
            allocation[good_periods] = good_capacity
            remaining = S - total_good

            other_periods = ~good_periods
            other_vol = v[other_periods]
            allocation[other_periods] = remaining * other_vol / other_vol.sum()
    else:
        # No good periods, just VWAP
        allocation = S * v / v.sum()

    return allocation


def participation_capped_vwap(df, S, max_participation=0.03):
    """
    Baseline: VWAP with participation cap

    Never exceed X% of bin volume
    """
    v = df['volume'].values

    # Initial VWAP allocation
    alloc = S * v / v.sum()

    # Cap at max participation
    max_alloc = v * max_participation

    # Iteratively redistribute excess
    for _ in range(10):
        excess_mask = alloc > max_alloc
        if not excess_mask.any():
            break

        excess = (alloc[excess_mask] - max_alloc[excess_mask]).sum()
        alloc[excess_mask] = max_alloc[excess_mask]

        # Redistribute to uncapped periods
        uncapped = ~excess_mask & (alloc < max_alloc)
        if uncapped.sum() > 0:
            uncapped_vol = v[uncapped]
            alloc[uncapped] += excess * uncapped_vol / uncapped_vol.sum()
            alloc[uncapped] = np.minimum(alloc[uncapped], max_alloc[uncapped])

    # Final rescale to hit target
    alloc = alloc * S / alloc.sum()
    return alloc


# ====================== SIMULATOR ======================
def simulate(df, allocation, impact, ofi_signal):
    prices = df['mid_price'].values
    volume = df['volume'].values
    spread = df['spread'].values
    ofi = df['ofi'].values

    arrival = prices[0]
    mkt_vwap = np.sum(prices * volume) / volume.sum()

    ofi_mult = ofi_signal.cost_multiplier(ofi)

    total_cost = 0
    total_shares = 0

    for i in range(len(df)):
        if allocation[i] > 0:
            impact_cost = impact.cost_per_share(allocation[i], volume[i], ofi_mult[i])
            exec_price = prices[i] + spread[i]/2 + impact_cost
            total_cost += exec_price * allocation[i]
            total_shares += allocation[i]

    avg_price = total_cost / total_shares

    return {
        'avg_price': avg_price,
        'vwap': mkt_vwap,
        'arrival': arrival,
        'is_vwap_bps': (avg_price / mkt_vwap - 1) * 10000,
        'is_vwap_dollars': (avg_price - mkt_vwap) * S,
        'is_arrival_bps': (avg_price / arrival - 1) * 10000,
    }


# ====================== ANALYSIS ======================
def analyze_allocation(df, allocation, name):
    """Analyze where the strategy allocates"""
    v = df['volume'].values
    ofi = df['ofi'].values

    # Normalize allocation to weights
    w = allocation / allocation.sum()
    vwap_w = v / v.sum()

    # Where does it overweight?
    diff = w - vwap_w
    overweight_mask = diff > 0.00001
    underweight_mask = diff < -0.00001

    print(f"\n  {name} allocation analysis:")
    print(f"    Overweight periods:  {overweight_mask.sum():4d} | avg vol: {v[overweight_mask].mean()/1e6:5.1f}M | avg OFI: {ofi[overweight_mask].mean():+.3f}")
    print(f"    Underweight periods: {underweight_mask.sum():4d} | avg vol: {v[underweight_mask].mean()/1e6:5.1f}M | avg OFI: {ofi[underweight_mask].mean():+.3f}")


# ====================== MAIN ======================
def main():
    print("="*70)
    print("VOLUME-AWARE OFI EXECUTION BACKTEST")
    print("="*70)

    df = load_data()
    print(f"Loaded {len(df):,} bins")

    train = df[df['is_train']]
    test = df[~df['is_train']]

    # Fit models
    impact = ImpactModel().fit(train)
    ofi_sig = OFISignal(lambda_ofi=0.5).fit(train['ofi'])

    print(f"Impact: eta={impact.eta:.4f}")
    print(f"OFI: mean={ofi_sig.mean:.4f}, std={ofi_sig.std:.4f}")

    # Strategies
    strategies = {
        'TWAP': lambda d: twap(d, S),
        'VWAP': lambda d: vwap(d, S),
        'VWAP (3% cap)': lambda d: participation_capped_vwap(d, S, 0.03),
        'OFI-VWAP Original': lambda d: ofi_vwap_original(d, S, ofi_sig, 0.5),
        'OFI-VWAP Multiplicative': lambda d: ofi_vwap_multiplicative(d, S, ofi_sig, 0.5),
        'OFI-VWAP Vol Floor': lambda d: ofi_vwap_volume_floor(d, S, ofi_sig, 0.5, 50),
        'OFI-VWAP Vol Scaled': lambda d: ofi_vwap_volume_scaled(d, S, ofi_sig, 0.5),
        'OFI Opportunistic': lambda d: ofi_opportunistic(d, S, ofi_sig, 0.5, 0.05),
    }

    dates = sorted(test['date'].unique())
    print(f"\nBacktesting on {len(dates)} test days...")

    all_results = []

    for date in dates:
        day = test[test['date'] == date].copy()
        print(f"\n{'='*70}")
        print(f"DATE: {date} ({len(day)} bins)")
        print(f"{'='*70}")

        for name, strat_fn in strategies.items():
            alloc = strat_fn(day)
            res = simulate(day, alloc, impact, ofi_sig)
            res['strategy'] = name
            res['date'] = date
            all_results.append(res)

            print(f"  {name:25s}: {res['is_vwap_bps']:+6.2f} bps  (${res['is_vwap_dollars']:+10,.0f})")

        # Analyze one day in detail
        if date == dates[0]:
            print("\n  --- Allocation Analysis (first day) ---")
            for name, strat_fn in strategies.items():
                if 'OFI' in name:
                    analyze_allocation(day, strat_fn(day), name)

    # Summary
    results_df = pd.DataFrame(all_results)

    print(f"\n{'='*70}")
    print("SUMMARY ACROSS ALL TEST DAYS")
    print(f"{'='*70}")

    print(f"\n{'Strategy':<28} {'Avg bps':>10} {'Std bps':>10} {'Total $':>14}")
    print("-" * 65)

    for strat in strategies.keys():
        s = results_df[results_df['strategy'] == strat]
        avg = s['is_vwap_bps'].mean()
        std = s['is_vwap_bps'].std()
        total = s['is_vwap_dollars'].sum()
        print(f"{strat:<28} {avg:>+10.2f} {std:>10.2f} {total:>+14,.0f}")

    # Best strategy
    print(f"\n{'='*70}")
    print("RANKING (by total $ vs VWAP)")
    print(f"{'='*70}")

    summary = results_df.groupby('strategy')['is_vwap_dollars'].sum().sort_values()
    for i, (strat, total) in enumerate(summary.items(), 1):
        print(f"  {i}. {strat:<28} ${total:>+12,.0f}")

    # Save
    results_df.to_csv(data_dir / 'execution_results_volume_aware.csv', index=False)
    print(f"\nSaved to execution_results_volume_aware.csv")

    return results_df


if __name__ == '__main__':
    results = main()

Target: Buy 100,000 shares
VOLUME-AWARE OFI EXECUTION BACKTEST
Loaded 51,480 bins
Impact: eta=0.6901
OFI: mean=0.0037, std=0.8260

Backtesting on 4 test days...

DATE: 2025-10-27 (2340 bins)
  TWAP                     :  +0.12 bps  ($      +820)
  VWAP                     :  +0.09 bps  ($      +582)
  VWAP (3% cap)            :  +0.09 bps  ($      +582)
  OFI-VWAP Original        :  -0.21 bps  ($    -1,448)
  OFI-VWAP Multiplicative  :  -0.51 bps  ($    -3,503)
  OFI-VWAP Vol Floor       :  -0.45 bps  ($    -3,094)
  OFI-VWAP Vol Scaled      :  +0.02 bps  ($      +160)
  OFI Opportunistic        :  -6.69 bps  ($   -45,709)

  --- Allocation Analysis (first day) ---

  OFI-VWAP Original allocation analysis:
    Overweight periods:  1480 | avg vol:  11.7M | avg OFI: -0.125
    Underweight periods:  746 | avg vol:  34.6M | avg OFI: +0.166

  OFI-VWAP Multiplicative allocation analysis:
    Overweight periods:   521 | avg vol:  27.2M | avg OFI: -0.648
    Underweight periods: 1267 | avg vo